# Paper Final Benchmark (Colab H100)

This notebook is for your paper-grade benchmark run.

It does four things:
1. Mounts Google Drive and enters the repo.
2. Creates a run-specific config from `configs/paper_final_500.toml`.
3. Builds graphs (if needed).
4. Runs 500-epoch benchmarking across `ff_layerwise`, `ff_e2e`, and `backprop`, then writes a paper summary table.

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/Forward-Risk-Manager") if IN_COLAB else Path.cwd()
if not ROOT.exists():
    raise FileNotFoundError(
        f"Repo root not found at {ROOT}. Update ROOT in this cell to your Drive path."
    )
os.chdir(ROOT)
print("repo root:", ROOT)
print("cwd:", Path.cwd())

In [ ]:
import shlex
import subprocess

def run(cmd: str) -> None:
    print("\n" + "=" * 120)
    print(cmd)
    print("=" * 120)
    subprocess.run(cmd, shell=True, check=True)

INSTALL_DEPS = True
if INSTALL_DEPS:
    run("python -m pip install --upgrade pip")
    run("python -m pip install -r requirements.txt")
    run("python -m pip install -e .")
else:
    print("Skipping dependency install (INSTALL_DEPS=False)")

In [ ]:
from datetime import datetime, timezone

TEMPLATE_CONFIG = ROOT / "configs" / "paper_final_500.toml"
assert TEMPLATE_CONFIG.exists(), f"Missing template config: {TEMPLATE_CONFIG}"

BASE_TOKEN = "runs/experiments/paper_final_500_TEMPLATE"
RUN_ID = f"paper_final_500_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID

for sub in ("data", "metrics", "plots", "logs", "models"):
    (RUN_ROOT / sub).mkdir(parents=True, exist_ok=True)

runtime_config = RUN_ROOT / "runtime_config.toml"
cfg_txt = TEMPLATE_CONFIG.read_text()
assert BASE_TOKEN in cfg_txt, f"Expected token {BASE_TOKEN!r} in template config"
cfg_txt = cfg_txt.replace(BASE_TOKEN, f"runs/experiments/{RUN_ID}")
runtime_config.write_text(cfg_txt)

print("run id:", RUN_ID)
print("runtime config:", runtime_config)
print("run root:", RUN_ROOT)

In [ ]:
RUN_BUILD_GRAPHS = True
if RUN_BUILD_GRAPHS:
    run(
        f"python -u scripts/build_graphs.py --config {shlex.quote(str(runtime_config))}"
    )
else:
    print("Skipping graph build (RUN_BUILD_GRAPHS=False)")

In [ ]:
BENCH_MODES = "ff_layerwise,ff_e2e,backprop"
run(
    f"python -u scripts/benchmark_training.py "
    f"--config {shlex.quote(str(runtime_config))} "
    f"--modes {BENCH_MODES}"
)

In [ ]:
benchmark_csv = RUN_ROOT / "metrics" / "benchmark.csv"
folds_csv = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"
summary_md = RUN_ROOT / "logs" / "paper_benchmark_summary.md"
summary_csv = RUN_ROOT / "metrics" / "paper_benchmark_summary.csv"
summary_json = RUN_ROOT / "logs" / "paper_benchmark_summary.json"

run(
    f"python -u scripts/paper_benchmark_summary.py "
    f"--benchmark {shlex.quote(str(benchmark_csv))} "
    f"--folds {shlex.quote(str(folds_csv))} "
    f"--out-md {shlex.quote(str(summary_md))} "
    f"--out-csv {shlex.quote(str(summary_csv))} "
    f"--out-json {shlex.quote(str(summary_json))}"
)

print("benchmark:", benchmark_csv)
print("folds:", folds_csv)
print("summary md:", summary_md)
print("summary csv:", summary_csv)
print("summary json:", summary_json)

In [ ]:
from IPython.display import Markdown, display

if summary_md.exists():
    display(Markdown(summary_md.read_text()))
else:
    print("Missing summary markdown:", summary_md)

print("\nArtifact check:")
for path in [benchmark_csv, folds_csv, summary_md, summary_csv, summary_json]:
    print(f"{path} -> exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")